# Phase 3 training on Google Colab (actions A2 and A8)

The laptop runs YOLOv8n (seed 0) and YOLO11n for the three categories. This notebook runs the rest on
Colab's free GPU, one model per run:

| Run | `MODEL` | `SEED` |
|---|---|---|
| Non-YOLO baseline (A8, needed) | `rtdetr-l` | 0 |
| Larger YOLO baseline (A8) | `yolov8s` | 0 |
| Seed variation of the proposed model (A2, optional) | `yolov8n` | 1 and 2 |

Everything is downloaded to this Colab machine's temporary disk; **nothing is saved to Google Drive**.
At the end you download one small zip (results plus the trained weights: 6 to 70 MB) to your computer.

Steps: `Runtime > Change runtime type > T4 GPU`, set the three values below, then `Runtime > Run all`.
If Colab disconnects during training, run all cells again: training resumes from the last epoch.

In [ ]:
CATEGORY = "pothole"   # "pothole", "garbage_litter" or "graffiti"
MODEL = "rtdetr-l"     # "rtdetr-l", "yolov8s" or "yolov8n"
SEED = 0
BATCH = 8 if MODEL.startswith("rtdetr") else 16   # RT-DETR-l does not fit batch 16 on a T4

In [ ]:
!nvidia-smi -L
import os
if not os.path.exists("/content/urbanfix"):
    !git clone --depth 1 https://github.com/pizzaparth/urbanfix.git /content/urbanfix
%cd /content/urbanfix/ai_model
!pip -q install ultralytics==8.4.173

## Data (from the public sources, onto this machine's disk)

In [ ]:
if CATEGORY == "pothole":
    !python scripts/fetch_rdd2022.py --countries India Japan Czech United_States China_MotorBike
elif CATEGORY == "garbage_litter":
    !python scripts/prepare_sources.py --only taco
    !python scripts/fetch_taco_unofficial.py
else:
    !python scripts/prepare_sources.py --only storm
!python scripts/fetch_coco_negatives.py
!python scripts/build_yolo_sets.py {CATEGORY}

## Train (40 epochs at most, early stopping after 10; checkpoint chosen on the validation split only)

In [ ]:
!python scripts/train.py {CATEGORY} --model {MODEL} --seed {SEED} --batch {BATCH} --device 0 --resume

## Evaluate once on the test and external sets, then score images for the complaint-level analysis

In [ ]:
TAG = f"{CATEGORY}__{MODEL}__s{SEED}"
!python scripts/evaluate.py --tags {TAG} --device 0
!python scripts/score_images.py --tags {TAG} --device 0
!python scripts/complaint_eval.py --tag {TAG}

## Download the results to your computer

Put the zip in `ai_model/` of your local repository and unzip it there (it adds files under `results/` and `weights/v2/`). The CPU latency is then remeasured on the laptop with `python scripts/evaluate.py --latency-only --tags <tag>`, so all models are timed on the same CPU.

In [ ]:
import shutil, os
out = f"/content/{TAG}"
for sub in ("train", "eval", "scores", "complaints"):
    os.makedirs(f"{out}/results/{sub}", exist_ok=True)
    src = f"results/{sub}/{TAG}.json"
    if os.path.exists(src):
        shutil.copy(src, f"{out}/results/{sub}/")
os.makedirs(f"{out}/weights/v2", exist_ok=True)
shutil.copy(f"weights/v2/{TAG}.pt", f"{out}/weights/v2/")
shutil.make_archive(out, "zip", out)
from google.colab import files
files.download(f"{out}.zip")